# Analiza wyników eksperymentów

In [2]:
import pandas as pd
import os

RESULTS_DIR = "../results/"
REQ_DIR_200 = RESULTS_DIR + "200req/"
REQ_DIR_3000 = RESULTS_DIR + "3000req/"

files_200_req = os.listdir(REQ_DIR_200)
files_3000_req = os.listdir(REQ_DIR_3000)

files_200_req_csv = ['200req/'+f for f in files_200_req if f.endswith('.csv')]
files_200_req_log = ['200req/'+f for f in files_200_req if f.endswith('.log')]
files_3000_req_csv = ['3000req/'+f for f in files_3000_req if f.endswith('.csv')]
files_3000_req_log = ['3000req/'+f for f in files_3000_req if f.endswith('.log')]

print("== Detected files ==")
print(f'200 req: {len(files_200_req_csv)} CSV, {len(files_200_req_log)} LOG')
print(f'3000 req: {len(files_3000_req_csv)} CSV, {len(files_3000_req_log)} LOG')

== Detected files ==
200 req: 8 CSV, 8 LOG
3000 req: 8 CSV, 8 LOG


## Wczytanie plików .csv i .log

In [3]:
import csv
from typing import List, Dict

def read_log_with_csv(path: str) -> pd.DataFrame:
    rows = []
    with open(path, newline='', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                continue
            rows.append(line)

    return pd.DataFrame(rows, columns=['data'])

In [4]:
df_200_req_csv = [(pd.read_csv(RESULTS_DIR + f), f) for f in files_200_req_csv]
df_3000_req_csv = [(pd.read_csv(RESULTS_DIR + f), f) for f in files_3000_req_csv]

df_200_req_log = [ (read_log_with_csv(RESULTS_DIR + f), f) for f in files_200_req_log]
df_3000_req_log = [ (read_log_with_csv(RESULTS_DIR + f), f) for f in files_3000_req_log]

In [5]:
print("== loaded csv dataframes ==")
for df, fname in df_200_req_csv + df_3000_req_csv:
    print(f"{fname}: \t{df.shape}")

== loaded csv dataframes ==
200req/docker_run_NB_fastapi_test_results.csv: 	(200, 6)
200req/docker_run_NB_ray_test_results.csv: 	(200, 6)
200req/docker_run_SVC_fastapi_test_results.csv: 	(200, 6)
200req/docker_run_SVC_ray_test_results.csv: 	(200, 6)
200req/local_run_NB_fastapi_test_results.csv: 	(200, 6)
200req/local_run_NB_ray_test_results.csv: 	(200, 6)
200req/local_run_SVC_fastapi_test_results.csv: 	(200, 6)
200req/local_run_SVC_ray_test_results.csv: 	(200, 6)
3000req/docker_run_NB_fastapi_test_results.csv: 	(3000, 10)
3000req/docker_run_NB_ray_test_results.csv: 	(3000, 10)
3000req/docker_run_SVC_fastapi_test_results.csv: 	(3000, 10)
3000req/docker_run_SVC_ray_test_results.csv: 	(3000, 10)
3000req/local_run_NB_fastapi_test_results.csv: 	(3000, 10)
3000req/local_run_NB_ray_test_results.csv: 	(3000, 10)
3000req/local_run_SVC_fastapi_test_results.csv: 	(3000, 10)
3000req/local_run_SVC_ray_test_results.csv: 	(3000, 10)


In [6]:
for df, fname in df_200_req_csv + df_3000_req_csv:
    splits = fname.split('_')
    ml_alg_type = splits[2]
    server_type = splits[3]
    env_type = splits[0]
    print(f"{fname}: \t{df.shape} \t{env_type} \t{ml_alg_type} \t{server_type}")

    df['env_type'] = env_type
    df['ml_alg_type'] = ml_alg_type
    df['server_type'] = server_type


200req/docker_run_NB_fastapi_test_results.csv: 	(200, 6) 	200req/docker 	NB 	fastapi
200req/docker_run_NB_ray_test_results.csv: 	(200, 6) 	200req/docker 	NB 	ray
200req/docker_run_SVC_fastapi_test_results.csv: 	(200, 6) 	200req/docker 	SVC 	fastapi
200req/docker_run_SVC_ray_test_results.csv: 	(200, 6) 	200req/docker 	SVC 	ray
200req/local_run_NB_fastapi_test_results.csv: 	(200, 6) 	200req/local 	NB 	fastapi
200req/local_run_NB_ray_test_results.csv: 	(200, 6) 	200req/local 	NB 	ray
200req/local_run_SVC_fastapi_test_results.csv: 	(200, 6) 	200req/local 	SVC 	fastapi
200req/local_run_SVC_ray_test_results.csv: 	(200, 6) 	200req/local 	SVC 	ray
3000req/docker_run_NB_fastapi_test_results.csv: 	(3000, 10) 	3000req/docker 	NB 	fastapi
3000req/docker_run_NB_ray_test_results.csv: 	(3000, 10) 	3000req/docker 	NB 	ray
3000req/docker_run_SVC_fastapi_test_results.csv: 	(3000, 10) 	3000req/docker 	SVC 	fastapi
3000req/docker_run_SVC_ray_test_results.csv: 	(3000, 10) 	3000req/docker 	SVC 	ray
3000req/

In [7]:
import re
import pandas as pd

NAME_MAPPING = {
    'd': 'docker',
    'nb': 'NB',
    'svc': 'SVC',
    'local': 'local',
    'docker': 'docker',
    'NB': 'NB',
    'SVC': 'SVC',
    'ray': 'ray',
    'fastapi': 'fastapi',
}

all_csv_files = df_200_req_csv + df_3000_req_csv

for logs, fname in df_200_req_log + df_3000_req_log:
    list_with_czas = [l for l in logs['data'] if 'Czas' in l]
    czas = []
    for line in list_with_czas:
        m = re.search(r'(\d+(?:\.\d+)?ms)', line)
        if m:
            czas.append(m.group(1))

    splits = re.split(r'[._/]', fname)

    current_tags = []
    for s in splits:
        if s in NAME_MAPPING:
            current_tags.append(NAME_MAPPING[s])

    if '200req' in splits: current_tags.append('200req')
    if '3000req' in splits: current_tags.append('3000req')

    matching_dfs = []

    for df_csv, fname_csv in all_csv_files:
        if 'csv' in fname_csv and all(tag in fname_csv for tag in current_tags):
            matching_dfs.append((df_csv, fname_csv))

    if matching_dfs:
        target_df, target_fname = matching_dfs[0]
        print(f"Znaleziono pasujący CSV: {target_fname}")

        if 'error_details' in target_df.columns:
            success_mask = target_df['error_details'].isna() | (target_df['error_details'] == "")
            num_successes = success_mask.sum()

            if num_successes == len(czas):
                target_df.loc[success_mask, 'log_czas'] = czas

        else:
            if len(target_df) == len(czas):
                target_df['log_czas'] = czas


Znaleziono pasujący CSV: 200req/docker_run_NB_fastapi_test_results.csv
Znaleziono pasujący CSV: 200req/docker_run_NB_ray_test_results.csv
Znaleziono pasujący CSV: 200req/docker_run_SVC_fastapi_test_results.csv
Znaleziono pasujący CSV: 200req/docker_run_SVC_ray_test_results.csv
Znaleziono pasujący CSV: 200req/local_run_NB_fastapi_test_results.csv
Znaleziono pasujący CSV: 200req/local_run_NB_ray_test_results.csv
Znaleziono pasujący CSV: 200req/local_run_SVC_fastapi_test_results.csv
Znaleziono pasujący CSV: 200req/local_run_SVC_ray_test_results.csv
Znaleziono pasujący CSV: 3000req/docker_run_NB_ray_test_results.csv
Znaleziono pasujący CSV: 3000req/docker_run_SVC_ray_test_results.csv
Znaleziono pasujący CSV: 3000req/docker_run_SVC_fastapi_test_results.csv
Znaleziono pasujący CSV: 3000req/docker_run_NB_fastapi_test_results.csv
Znaleziono pasujący CSV: 3000req/local_run_SVC_ray_test_results.csv
Znaleziono pasujący CSV: 3000req/local_run_NB_ray_test_results.csv
Znaleziono pasujący CSV: 3000re

In [8]:
df_200_req_csv[0][0]['log_czas']

0      48.11ms
1      48.20ms
2      48.28ms
3      48.38ms
4      48.47ms
        ...   
195    29.21ms
196    29.34ms
197    29.50ms
198    29.66ms
199    29.81ms
Name: log_czas, Length: 200, dtype: object

In [9]:
new_processed_df = []

for df, fname in df_200_req_csv + df_3000_req_csv:

    if 'log_czas' in df.columns:
        if df['log_czas'].dtype == object:
            df['log_czas'] = df['log_czas'].str.replace('ms', '')
            df['log_czas'] = pd.to_numeric(df['log_czas'], errors='coerce')

    record = {'filename': fname,
              'model_used': df['model_used'][0] if 'model_used' in df.columns else None,
              'server_type': df['server_type'][0] if 'server_type' in df.columns else None,
              'env_type': str(df['env_type'][0]).split('/')[1] if 'env_type' in df.columns else None,
              'num_requests': len(df),
              'num_success': len(df[df['status'] == 'success']),
              'num_failure': len(df[df['status'] != 'success']),
              'avg_duration_seconds': df['duration_seconds'].mean() if 'duration_seconds' in df.columns else None,
              'avg_log_czas_ms': df['log_czas'].mean() if 'log_czas' in df.columns else None}

    print('== loaded csv dataframes ==')
    print(f"uzyty model: {record['model_used']}")
    print(f"typ servera: {record['server_type']}")
    print(f"rodzaj srodowiska: {record['env_type']}")
    print(f"liczba zapytań: {record['num_requests']}")
    print(f"liczba zapytań zakończona sukcesem: {record['num_success']}")
    print(f"liczba zapytań nie zakończona sukcesem: {record['num_failure']}")
    if 'duration_seconds' in df.columns:
        print(f"średni czas przetwarzania (s): {record['avg_duration_seconds']}")
    if 'log_czas' in df.columns:
        print(f"średni czas przetwarzania przez serwer (ms): {record['avg_log_czas_ms']}")

    new_processed_df.append(record)


== loaded csv dataframes ==
uzyty model: MultinomialNB
typ servera: fastapi
rodzaj srodowiska: docker
liczba zapytań: 200
liczba zapytań zakończona sukcesem: 200
liczba zapytań nie zakończona sukcesem: 0
średni czas przetwarzania (s): 0.228422
średni czas przetwarzania przez serwer (ms): 59.4654
== loaded csv dataframes ==
uzyty model: MultinomialNB
typ servera: ray
rodzaj srodowiska: docker
liczba zapytań: 200
liczba zapytań zakończona sukcesem: 200
liczba zapytań nie zakończona sukcesem: 0
średni czas przetwarzania (s): 0.9773394999999999
średni czas przetwarzania przez serwer (ms): 16.277449999999998
== loaded csv dataframes ==
uzyty model: SVC
typ servera: fastapi
rodzaj srodowiska: docker
liczba zapytań: 200
liczba zapytań zakończona sukcesem: 200
liczba zapytań nie zakończona sukcesem: 0
średni czas przetwarzania (s): 2.0903110000000003
średni czas przetwarzania przez serwer (ms): 1053.2984
== loaded csv dataframes ==
uzyty model: SVC
typ servera: ray
rodzaj srodowiska: docker
li

In [10]:
new_processed_df = pd.DataFrame(new_processed_df)
new_processed_df = new_processed_df.rename_axis(index=None)

In [11]:
from tabulate import tabulate

cols = new_processed_df.columns

print(new_processed_df[cols[1:]].to_markdown(tablefmt="grid"))

+----+---------------+---------------+------------+----------------+---------------+---------------+------------------------+-------------------+
|    | model_used    | server_type   | env_type   |   num_requests |   num_success |   num_failure |   avg_duration_seconds |   avg_log_czas_ms |
+====+===============+===============+============+================+===============+===============+========================+===================+
|  0 | MultinomialNB | fastapi       | docker     |            200 |           200 |             0 |               0.228422 |           59.4654 |
+----+---------------+---------------+------------+----------------+---------------+---------------+------------------------+-------------------+
|  1 | MultinomialNB | ray           | docker     |            200 |           200 |             0 |               0.977339 |           16.2774 |
+----+---------------+---------------+------------+----------------+---------------+---------------+------------------------

In [ ]:
new_df_cut = new_processed_df[cols[1:]]
new_df_cut.drop('num_failure', axis=1, inplace=True)
print(new_df_cut[new_df_cut['model_used'] == 'MultinomialNB'][new_df_cut.columns[1:]].to_markdown(tablefmt="grid"))

+----+---------------+------------+----------------+---------------+------------------------+-------------------+
|    | server_type   | env_type   |   num_requests |   num_success |   avg_duration_seconds |   avg_log_czas_ms |
+====+===============+============+================+===============+========================+===================+
|  0 | fastapi       | docker     |            200 |           200 |               0.228422 |           59.4654 |
+----+---------------+------------+----------------+---------------+------------------------+-------------------+
|  1 | ray           | docker     |            200 |           200 |               0.977339 |           16.2774 |
+----+---------------+------------+----------------+---------------+------------------------+-------------------+
|  4 | fastapi       | local      |            200 |           200 |               0.190382 |           32.0022 |
+----+---------------+------------+----------------+---------------+--------------------

C:\Users\admin\AppData\Local\Temp\ipykernel_28616\3197705083.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df_cut.drop('num_failure', axis=1, inplace=True)


In [15]:
new_df_cut = new_processed_df[cols[1:]]
new_df_cut.drop('num_failure', axis=1, inplace=True)
print(new_df_cut[new_df_cut['model_used'] == 'SVC'][new_df_cut.columns[1:]].to_markdown(tablefmt="grid"))

+----+---------------+------------+----------------+---------------+------------------------+-------------------+
|    | server_type   | env_type   |   num_requests |   num_success |   avg_duration_seconds |   avg_log_czas_ms |
+====+===============+============+================+===============+========================+===================+
|  2 | fastapi       | docker     |            200 |           200 |                2.09031 |         1053.3    |
+----+---------------+------------+----------------+---------------+------------------------+-------------------+
|  3 | ray           | docker     |            200 |           200 |                2.13235 |           53.4757 |
+----+---------------+------------+----------------+---------------+------------------------+-------------------+
|  6 | fastapi       | local      |            200 |           200 |                2.61923 |          831.904  |
+----+---------------+------------+----------------+---------------+--------------------

C:\Users\admin\AppData\Local\Temp\ipykernel_28616\151256248.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df_cut.drop('num_failure', axis=1, inplace=True)
